# 🏦 Banking Document Search Agent Tutorial 📄

Welcome to the **Banking Document Search Agent** tutorial! We'll use **Microsoft Foundry** SDKs to build an assistant that can:

1. **Upload** banking policy documents and loan guidelines into a vector store.
2. **Create an Agent** with a **File Search** tool.
3. **Search** these documents for loan policies, banking regulations, and compliance information.
4. **Answer** customer and employee questions about banking products and procedures.

### ⚠️ Important Financial Disclaimer ⚠️
> **All financial information in this notebook is for general educational purposes only and is not a substitute for professional financial or legal advice.** Always consult with qualified banking professionals and compliance officers for official guidance.

## Prerequisites

### 🔐 Required Roles
1. **Azure AI Developer** on your Microsoft Foundry project.
2. **Storage Blob Data Contributor** on the project's Storage account.
3. If standard agent setup is used with your own Search resource, also ensure you have **Cognitive Search Data Contributor** on that resource.

### 🌐 Storage Account Networking Configuration
The file upload to vector stores requires network access to the storage account. If you encounter a **403 Forbidden** error during file upload, you need to configure the storage account networking:

1. Go to **Azure Portal** → Navigate to your AI Foundry project's **Storage Account**
2. Go to **Networking** under **Security + networking**
3. Under **Public network access**, select **"Enabled from all networks"**
4. Click **Save** and wait 1-2 minutes for changes to propagate

> ⚠️ **Important**: For workshops/testing, enabling "from all networks" is the most reliable option. Other configurations (selected networks, adding IP addresses, resource instances) may not work reliably for file uploads. For production environments, consult your security team for appropriate network configurations.

## Let's Get Searching!
We'll show you how to upload sample banking documents, create a vector store for them, then spin up an agent that can search for loan policies, interest rates, and compliance guidelines. Enjoy!

## 🔐 Authentication Setup

Before running the next cell, make sure you're authenticated with Azure CLI. Run this command in your terminal:

```bash
az login --use-device-code
```

This will provide you with a device code and URL to authenticate in your browser, which is useful for:
- Remote development environments
- Systems without a default browser
- Corporate environments with strict security policies

After successful authentication, you can proceed with the notebook cells below.

## 1. Initial Setup
Here we import needed libraries, load environment variables from `.env`, and initialize our **AIProjectClient**. Let's do this! 🎉

In [ ]:
import os
from pathlib import Path

from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import FileSearchTool, PromptAgentDefinition
from azure.identity import DefaultAzureCredential
from dotenv import find_dotenv, load_dotenv


dotenv_path = find_dotenv(usecwd=True)
if not dotenv_path:
    raise FileNotFoundError("Could not find a .env file in the current directory or its parents.")
load_dotenv(dotenv_path)

project_endpoint = os.getenv("AI_FOUNDRY_PROJECT_ENDPOINT")
model_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")
required_settings = {
    "AI_FOUNDRY_PROJECT_ENDPOINT": project_endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": model_name,
}
missing_settings = [name for name, value in required_settings.items() if not value]
if missing_settings:
    raise RuntimeError(f"Missing required environment variables: {', '.join(missing_settings)}")

credential = DefaultAzureCredential()
project_client = AIProjectClient(endpoint=project_endpoint, credential=credential)
openai_client = project_client.get_openai_client()
print("AIProjectClient and OpenAI client initialized")

## 2. Prepare Sample Banking Documents 📄💼
We'll create sample markdown files for loan policies and banking guidelines. Then we'll store them in a vector store for searching.

In [ ]:
def create_sample_files():
    loan_policies_md = """# Loan Policies and Guidelines

## Mortgage Loan Requirements
### Eligibility Criteria
- Minimum credit score: 620 for conventional loans, 580 for FHA loans
- Debt-to-income ratio: Maximum 43% for most loan programs
- Employment history: Minimum 2 years of stable employment
- Down payment: Minimum 3% for conventional, 3.5% for FHA

### Interest Rate Tiers
| Credit Score | Rate Adjustment |
|--------------|-----------------|
| 760+         | Best available rate |
| 700-759      | +0.25% |
| 660-699      | +0.50% |
| 620-659      | +0.75% |

## Auto Loan Guidelines
- Maximum loan-to-value: 125% for new vehicles, 100% for used
- Maximum term: 84 months for new, 72 months for used vehicles
- Vehicle age restrictions: Maximum 7 years old for used vehicles
- Required documentation: Proof of income, insurance verification

## Personal Loan Policies
- Unsecured loans up to $50,000
- Terms from 12 to 60 months
- Fixed interest rates based on creditworthiness
- No prepayment penalties

## Business Loan Requirements
- Minimum 2 years in business
- Annual revenue documentation required
- Business plan for loans over $100,000
- Personal guarantee may be required
"""

    compliance_guidelines_md = """# Banking Compliance Guidelines

## Truth in Lending Act (TILA) Requirements
- APR disclosure must be provided within 3 business days
- All fees must be clearly itemized
- Right to rescission for home equity loans (3 business day period)
- Clear disclosure of payment schedules

## Fair Lending Practices
- Equal Credit Opportunity Act compliance required
- No discrimination based on race, religion, national origin, sex, marital status, age
- Consistent underwriting criteria for all applicants
- Documentation of all lending decisions

## Know Your Customer (KYC) Requirements
- Government-issued ID verification
- Address verification through utility bill or bank statement
- Source of funds documentation for large transactions
- Enhanced due diligence for high-risk customers

## Anti-Money Laundering (AML) Compliance
- Currency Transaction Reports for transactions over $10,000
- Suspicious Activity Reports when warranted
- Customer identification program implementation
- Ongoing transaction monitoring

## Data Privacy Requirements
- GLBA compliance for customer information protection
- Secure data storage and transmission
- Customer consent for information sharing
- Annual privacy notice distribution
"""

    # Save to local files
    loan_filename = os.environ.get("LOAN_POLICIES_FILENAME", "loan_policies.md")
    compliance_filename = os.environ.get("COMPLIANCE_FILENAME", "compliance_guidelines.md")
    
    with open(loan_filename, "w", encoding="utf-8") as f:
        f.write(loan_policies_md)
    with open(compliance_filename, "w", encoding="utf-8") as f:
        f.write(compliance_guidelines_md)

    print(f"📄 Created sample banking documents: {loan_filename}, {compliance_filename}")
    return [loan_filename, compliance_filename]

sample_files = create_sample_files()

#### ✨ Note on Search Permissions
When creating the vector store, you must also have **Cognitive Search Data Contributor** role on your Azure AI Search resource (if you're using the standard agent setup with your own Search resource). Missing this role will often cause a **Forbidden** error. See [Authentication Setup](../../1-introduction/1-authentication.ipynb#4-add-agent-service-permissions) for details on configuring permissions.


## 3. Create a Vector Store for Banking Documents 📚
We'll upload our banking policy documents and group them into a single vector store for searching. This allows the agent to find relevant policy information quickly.

In [ ]:
def create_vector_store(files, store_name="banking_documents"):
    """Create a vector store and upload each source file."""
    vector_store = openai_client.vector_stores.create(name=store_name)
    print(f"Vector store created: {vector_store.id}")

    uploaded_file_ids = []
    try:
        for file_path in map(Path, files):
            print(f"Uploading {file_path}...")
            with file_path.open("rb") as file_stream:
                uploaded_file = openai_client.vector_stores.files.upload_and_poll(
                    vector_store_id=vector_store.id,
                    file=file_stream,
                )
            uploaded_file_ids.append(uploaded_file.id)
            print(f"Uploaded file: {uploaded_file.id}")
    except Exception:
        openai_client.vector_stores.delete(vector_store.id)
        raise

    return vector_store, uploaded_file_ids


vector_store, file_ids = create_vector_store(sample_files, "banking_policies_store")

## 4. Create the Banking Document Search Agent 🔎
We use a **FileSearchTool** pointing to our newly created vector store, then create the Agent with instructions about banking policies, loan guidelines, and compliance information.

In [ ]:
def create_banking_document_agent(vector_store_id):
    """Create a banking document search agent backed by the vector store."""
    file_search_tool = FileSearchTool(vector_store_ids=[vector_store_id])
    agent = project_client.agents.create_version(
        agent_name="banking-document-search-agent",
        definition=PromptAgentDefinition(
            model=model_name,
            instructions="""
            You are a Banking Document Search Agent with access to loan policies and compliance guidelines.

            1. Search the uploaded documents before answering.
            2. Cite the relevant document or policy section when possible.
            3. Focus on loan requirements, interest rates, and compliance guidelines.
            4. Explain banking terms in clear, customer-friendly language.
            5. Include appropriate financial and regulatory disclaimers.
            6. Encourage users to consult qualified banking professionals for official guidance.
            """,
            tools=[file_search_tool],
        ),
        description="Banking document search agent for policy and compliance queries.",
    )
    print(f"Created agent {agent.name}, version {agent.version}")
    return agent


banking_agent = create_banking_document_agent(vector_store.id)

## 5. Searching Banking Documents 🏋️👩‍💼
We'll create a new conversation thread and ask queries like "What credit score do I need for a mortgage?" or "What are the KYC requirements?" The agent will search our banking documents to find relevant information.

In [ ]:
def create_search_conversation():
    conversation = openai_client.conversations.create()
    print(f"Created search conversation: {conversation.id}")
    return conversation


def ask_search_question(agent, user_question, conversation_id):
    """Ask a document-grounded question through the Responses API."""
    print(f"Searching: {user_question}")
    response = openai_client.responses.create(
        conversation=conversation_id,
        extra_body={
            "agent_reference": {
                "type": "agent_reference",
                "name": agent.name,
                "version": agent.version,
            }
        },
        input=user_question,
    )
    if not response.output_text:
        raise RuntimeError(f"Response {response.id} did not contain output text.")
    print(response.output_text)
    return response


search_conversation = create_search_conversation()
queries = [
    "What is the minimum credit score required for a mortgage loan?",
    "What are the KYC requirements for new customers?",
    "What documentation is needed for a business loan over $100,000?",
]
search_responses = [
    ask_search_question(banking_agent, query, search_conversation.id)
    for query in queries
]

## 6. Cleanup & Best Practices 🧹
We'll optionally remove the vector store, the uploaded files, and the agent. In a production environment, you might keep them around longer. Meanwhile, here are some tips:

1. **Resource Management**
   - Keep files grouped by category, regularly prune old or irrelevant files.
   - Clear out test agents or vector stores once you're done.

2. **Search Queries**
   - Provide precise or multi-part queries.
   - Consider synonyms or alternative keywords ("gluten-free" vs "celiac").
   
3. **Health Information**
   - Always disclaim that you are not a medical professional.
   - Encourage users to see doctors for specific diagnoses.

4. **Performance**
   - Keep an eye on vector store size.
   - Evaluate search accuracy with `azure-ai-evaluation`!


In [ ]:
def cleanup_all():
    """Delete resources created by this notebook and close SDK clients."""
    cleanup_errors = []

    cleanup_operations = [
        (
            f"conversation {search_conversation.id}",
            lambda: openai_client.conversations.delete(search_conversation.id),
        ),
        (
            f"agent {banking_agent.name} version {banking_agent.version}",
            lambda: project_client.agents.delete_version(
                agent_name=banking_agent.name,
                agent_version=banking_agent.version,
            ),
        ),
        (
            f"vector store {vector_store.id}",
            lambda: openai_client.vector_stores.delete(vector_store.id),
        ),
    ]

    for resource_name, delete_resource in cleanup_operations:
        try:
            delete_resource()
            print(f"Deleted {resource_name}")
        except Exception as error:
            cleanup_errors.append(f"{resource_name}: {error}")

    for file_path in map(Path, sample_files):
        try:
            file_path.unlink(missing_ok=True)
            print(f"Deleted local file {file_path}")
        except Exception as error:
            cleanup_errors.append(f"local file {file_path}: {error}")

    openai_client.close()
    project_client.close()
    credential.close()

    if cleanup_errors:
        raise RuntimeError("Cleanup failures:\n" + "\n".join(cleanup_errors))


cleanup_all()

# Congratulations! 🎉

You've successfully completed the **Banking Document Search Agent** tutorial! Here's what was accomplished:

## ✅ **What We Built**

### **🔍 File Search Agent**
- Created an AI agent with **FileSearchTool** capabilities
- Enabled the agent to search through uploaded banking documents using semantic search
- Configured banking-focused instructions with appropriate financial disclaimers

### **📚 Key Features Demonstrated**

1. **📄 Vector Store & File Upload**
   - Created a vector store using `openai_client.vector_stores.create()`
   - Uploaded files directly to vector store using `openai_client.vector_stores.files.upload_and_poll()`
   - Polled for completion status

2. **🔎 Semantic Document Search**
   - Agent searches through loan policies and compliance guidelines
   - Provides relevant answers based on uploaded document contents
   - Uses `FileSearchTool` class from `azure.ai.projects.models`

3. **🏦 Banking-Focused Responses**
   - Agent answered questions about mortgage requirements, KYC policies, and business loans
   - Provided responsible AI disclaimers about financial advice
   - Referenced content from uploaded policy documents

4. **🧹 Resource Management**
   - Properly cleaned up vector stores and agents
   - Demonstrated best practices for resource lifecycle management


## 🎯 **Key Concepts Learned**

- **FileSearchTool**: The proper class from `azure.ai.projects.models` for file search
- **Vector Stores**: Create empty first, then upload files with `upload_and_poll()`
- **Polling Pattern**: Wait for vector store status to be "completed"
- **Semantic Search**: Agents find relevant content even when exact words don't match
- **Responsible AI**: Always include financial disclaimers for banking-related content

## 🔍 **API Methods Reference**

| Method | Description |
|--------|-------------|
| `openai_client.vector_stores.create()` | Create empty vector store |
| `openai_client.vector_stores.files.upload_and_poll()` | Upload file and wait for processing |
| `openai_client.vector_stores.retrieve()` | Check vector store status |
| `FileSearchTool(vector_store_ids=[...])` | Create file search tool |
| `project_client.agents.create_version()` | Create agent with tools |
| `openai_client.conversations.create()` | Create conversation |
| `openai_client.responses.create()` | Get agent response |

## 💡 **Best Practices Recap**

1. **Vector Store Pattern** - Create empty vector store first, then upload files to it
2. **Use FileSearchTool Class** - Import from `azure.ai.projects.models`
3. **Polling** - Always wait for vector store status to be "completed"
4. **Banking Content** - Always include financial disclaimers
5. **Resource Cleanup** - Delete agents first, then vector stores
6. **Error Handling** - Check for 403 errors indicating missing permissions

## 🔧 **Troubleshooting Guide**

**If you get a 403 error during file upload:**
1. ✅ Ensure you have **Storage Blob Data Contributor** role on the project's storage account
2. ✅ Go to Azure Portal → Your AI Foundry project → Access control (IAM)
3. ✅ Add role assignment for your user account
4. ✅ Wait a few minutes for the role to propagate

**If vector store creation fails:**
1. ✅ Use the correct pattern: create empty store, then upload files
2. ✅ Use `upload_and_poll()` instead of separate upload methods
3. ✅ Check the vector store status after uploads

## 📚 **Reference**

- [Azure AI Agents Documentation](https://learn.microsoft.com/azure/ai-services/agents/)

---

*Happy document searching!* 🔍💼